# Compare Ecowitt data AW005 with reference sensors from DGA, Chile

This notebook loads **Ecowitt** data from the repo., **reference sensor** file from the repo., and compares selected variables, computes metrics, and creates plots.

In [71]:
#@title 1) Install / import packages
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [73]:
#@title 2) User configuration
REPO_URL = "https://github.com/paulmunozpauta/VLIR_SI_EC_2025-2027"
REPO_BRANCH = "main"
REPO_DIR = "/content/VLIR_SI_EC_2025-2027"
# Clone repo only if not already present
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH $REPO_URL $REPO_DIR
else:
    print("Repository already cloned")
ECOWITT_FILE = "datasets/AW005.csv"
REFERENCE_FILE = "Validation_meteo/DGA_ChillanUdeC.xlsx"

Repository already cloned


In [74]:
#@title 3) Load Ecowitt data

ecowitt_path = os.path.join(
    REPO_DIR,
    ECOWITT_FILE
)

ecowitt_columns = [

    "Date",

    "air_temperature_c",

    "relative_humidity_pct",

    "dew_point_c",

    "feels_like_c",

    "wind_speed_ms",

    "wind_gust_ms",

    "wind_direction_deg",

    "atmospheric_pressure_hpa",

    "rain_rate_mm_hr",

    "daily_rain_mm",

    "solar_radiation_w_m2",

    "uv_index",

    "indoor_temperature_c",

    "indoor_relative_humidity_pct"
]

ecowitt = pd.read_csv(
    ecowitt_path,
    names=ecowitt_columns,
    header=0
)

# Convert datetime
ecowitt["Date"] = pd.to_datetime(
    ecowitt["Date"],
    errors="coerce",
    utc=True
)

# Remove invalid dates
ecowitt = ecowitt.dropna(subset=["Date"])

# Make datetime index
ecowitt = ecowitt.set_index("Date")

# Convert numeric columns
for col in ecowitt.columns:

    ecowitt[col] = pd.to_numeric(
        ecowitt[col],
        errors="coerce"
    )

# Sort chronologically
ecowitt = ecowitt.sort_index()

print(ecowitt.shape)

display(ecowitt.head())

(3374, 14)


,air_temperature_c,relative_humidity_pct,dew_point_c,feels_like_c,wind_speed_ms,wind_gust_ms,wind_direction_deg,atmospheric_pressure_hpa,rain_rate_mm_hr,daily_rain_mm,solar_radiation_w_m2,uv_index,indoor_temperature_c,indoor_relative_humidity_pct
Date,,,,,,,,,,,,,,
2025-12-22 19:01:17.852000+00:00,26.3,40.0,11.6,26.3,0.0,0.0,127.0,29.91,0.0,0.8,2.1,0.0,26.1,40.0
2025-12-22 20:01:17.232000+00:00,26.9,36.0,10.6,26.6,0.0,0.0,245.0,29.90,0.0,0.3,1.9,0.0,26.7,37.0
2025-12-22 21:01:18.421000+00:00,26.8,36.0,10.5,26.5,0.0,0.0,245.0,29.91,0.0,0.3,1.7,0.0,26.4,36.0
2025-12-22 22:01:14.613000+00:00,26.5,35.0,9.8,26.5,0.0,0.0,245.0,29.91,0.0,0.3,1.4,0.0,26.1,35.0
2025-12-22 23:00:30.608000+00:00,26.0,36.0,9.8,26.0,0.0,0.0,245.0,29.94,0.0,0.3,1.0,0.0,25.5,36.0


In [75]:
#@title 4) Load reference data

reference_path = os.path.join(
    REPO_DIR,
    REFERENCE_FILE
)

reference = pd.read_excel(
    reference_path,
    header=1
)

# Remove unwanted column
reference = reference.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# Rename datetime column
reference = reference.rename(columns={
    "Unnamed: 1": "Date"
})

# Convert datetime
reference["Date"] = pd.to_datetime(
    reference["Date"],
    errors="coerce"
)

# Remove invalid dates
reference = reference.dropna(subset=["Date"])

# Make datetime index
reference = reference.set_index("Date")

# Rename columns
reference = reference.rename(columns={

    "Pptación Acum. (mm)": "precipitation_mm",

    "Temp.del Aire (ºC)": "air_temperature_c",

    "Humedad (%)": "relative_humidity_pct",

    "Rad.Solar (Watt/m2)": "solar_radiation_w_m2",

    "Presión Atmosférica (mb)": "atmospheric_pressure_hpa",

    "Pp. acum. disdrometro (mm)": "disdrometer_precipitation_mm",

    "Direccion del Viento (grados)": "wind_direction_deg",

    "Velocidad del Vto. (m/s)": "wind_speed_ms",

    "Calidad de señal (db)": "signal_quality_db",

    "Pp. instan. disdrometro (mm)": "disdrometer_instantaneous_rainfall_mm",

    "Tipo Pptacion disdrometro ( )": "disdrometer_precipitation_type"

})

# Convert numeric columns
for col in reference.columns:

    if col != "disdrometer_precipitation_type":

        reference[col] = pd.to_numeric(
            reference[col],
            errors="coerce"
        )

# Sort chronologically
reference = reference.sort_index()

print(reference.shape)

display(reference.head())

(4310, 11)


,precipitation_mm,air_temperature_c,relative_humidity_pct,solar_radiation_w_m2,atmospheric_pressure_hpa,disdrometer_precipitation_mm,wind_direction_deg,wind_speed_ms,signal_quality_db,disdrometer_instantaneous_rainfall_mm,disdrometer_precipitation_type
Date,,,,,,,,,,,
2026-02-13 00:00:00,5.7,18.1,37.9,0.0,1002.6,620.3,218.1,2.3,-113.0,0.0,0
2026-02-13 00:30:00,5.7,17.3,41.7,0.0,1002.5,620.3,182.9,1.9,-113.0,0.0,0
2026-02-13 01:00:00,5.7,16.7,45.5,0.0,1002.4,620.3,218.6,1.7,-113.0,0.0,0
2026-02-13 01:30:00,5.7,16.0,48.9,0.0,1002.3,620.3,232.3,1.9,-113.0,0.0,0
2026-02-13 02:00:00,5.7,15.6,51.7,0.0,1002.2,620.3,231.8,1.0,-113.0,0.0,0


In [81]:
#@title 5) Resample reference hourly

agg_rules = {}

for col in reference.columns:

    # Precipitation → sum
    if col in [
        "precipitation_mm",
        "disdrometer_precipitation_mm",
        "disdrometer_instantaneous_rainfall_mm"
    ]:

        agg_rules[col] = "sum"

    # Precipitation type → last value
    elif col == "disdrometer_precipitation_type":

        agg_rules[col] = "last"

    # Everything else → mean
    else:

        agg_rules[col] = "mean"

reference_hourly = reference.resample("1H").agg(agg_rules)

# Remove fully empty rows
reference_hourly = reference_hourly.dropna(how="all")

print(reference_hourly.shape)

display(reference_hourly.head())

(2157, 11)


/tmp/ipykernel_674/3388510715.py:26: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  reference_hourly = reference.resample("1H").agg(agg_rules)


,precipitation_mm,air_temperature_c,relative_humidity_pct,solar_radiation_w_m2,atmospheric_pressure_hpa,disdrometer_precipitation_mm,wind_direction_deg,wind_speed_ms,signal_quality_db,disdrometer_instantaneous_rainfall_mm,disdrometer_precipitation_type
Date,,,,,,,,,,,
2026-02-13 00:00:00,11.4,17.70,39.80,0.0,1002.55,1240.6,200.50,2.1,-113.0,0.0,0
2026-02-13 01:00:00,11.4,16.35,47.20,0.0,1002.35,1240.6,225.45,1.8,-113.0,0.0,0
2026-02-13 02:00:00,11.4,15.25,53.30,0.0,1002.15,1240.6,221.05,0.8,-113.0,0.0,0
2026-02-13 03:00:00,11.4,14.55,57.80,0.0,1001.75,1240.6,216.25,1.4,-113.0,0.0,0
2026-02-13 04:00:00,11.4,14.05,63.15,0.0,1001.50,1240.6,194.25,1.0,-113.0,0.0,0


In [82]:
#@title 6) Merge Ecowitt and reference

# Ecowitt already UTC
ecowitt.index = pd.to_datetime(
    ecowitt.index,
    utc=True
)

# Reference is Chile local time
reference_hourly.index = (
    pd.to_datetime(reference_hourly.index)
    .tz_localize(
        "America/Santiago",
        ambiguous=False
    )
    .tz_convert("UTC")
)

comparison = pd.merge_asof(
    ecowitt.sort_index(),
    reference_hourly.sort_index(),
    left_index=True,
    right_index=True,
    direction="nearest",
    tolerance=pd.Timedelta("30min"),
    suffixes=("_ecowitt", "_reference")
)

print(comparison.shape)

display(comparison.head(10))

(3374, 25)


,air_temperature_c_ecowitt,relative_humidity_pct_ecowitt,dew_point_c,feels_like_c,wind_speed_ms_ecowitt,wind_gust_ms,wind_direction_deg_ecowitt,atmospheric_pressure_hpa_ecowitt,rain_rate_mm_hr,daily_rain_mm,...,air_temperature_c_reference,relative_humidity_pct_reference,solar_radiation_w_m2_reference,atmospheric_pressure_hpa_reference,disdrometer_precipitation_mm,wind_direction_deg_reference,wind_speed_ms_reference,signal_quality_db,disdrometer_instantaneous_rainfall_mm,disdrometer_precipitation_type
Date,,,,,,,,,,,,,,,,,,,,,
2025-12-22 19:01:17.852000+00:00,26.3,40.0,11.6,26.3,0.0,0.0,127.0,29.91,0.0,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-22 20:01:17.232000+00:00,26.9,36.0,10.6,26.6,0.0,0.0,245.0,29.90,0.0,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-22 21:01:18.421000+00:00,26.8,36.0,10.5,26.5,0.0,0.0,245.0,29.91,0.0,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-22 22:01:14.613000+00:00,26.5,35.0,9.8,26.5,0.0,0.0,245.0,29.91,0.0,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-22 23:00:30.608000+00:00,26.0,36.0,9.8,26.0,0.0,0.0,245.0,29.94,0.0,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-23 00:00:34.026000+00:00,25.0,38.0,9.7,25.0,0.0,0.0,245.0,29.96,0.0,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-23 01:00:39.429000+00:00,23.3,41.0,9.3,23.3,0.0,0.0,245.0,29.98,0.0,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-23 02:00:42.979000+00:00,21.7,45.0,9.3,21.7,0.0,0.0,245.0,29.99,0.0,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-23 03:00:32.135000+00:00,20.5,49.0,9.4,20.5,0.0,0.0,245.0,29.99,0.0,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
